# WAFER-GUARD · Fab-7 Excursion Detector (repaired)

Repaired version of `original/bnypi.ipynb`. The original reported F1 0.99, but on a SMOTE-balanced test set that contained synthetic copies of training wafers. Here everything is fit on the past only and scored on a time-ordered holdout at natural prevalence. Findings are in `AUDIT.md`, the evidence for each one is in section 8.

### Data dictionary: `secom_telemetry_modified.csv`

| column | description |
|---|---|
| `sensor_0 … sensor_589` | 590 in-line process / metrology signals per wafer lot (anonymised) |
| `label` | electrical-test outcome: `-1` = pass, `+1` = fail |
| `timestamp` | lot start time (the fab runs continuously; equipment ages, recipes drift) |

PS-L legacy probe station: `sensor_0 == 0.0` is a sentinel and `sensor_1` holds firmware codes 100 / 200 / 300 on those wafers.

## 0 · Environment

In [1]:
# !pip install -r requirements.txt

In [2]:
import os, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (f1_score, precision_score, recall_score, roc_auc_score,
                             average_precision_score, confusion_matrix, precision_recall_curve)
from imblearn.over_sampling import SMOTE
import joblib

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)   # nanstd of all-NaN columns

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(4)   # small model, using every core was much slower on my machine

HP = dict(batch_size=64, epochs=40, lr=1e-3, weight_decay=1e-5,
          hidden=256, latent=32, lambda_clf=1.0,
          split=(0.6, 0.2, 0.2), max_missing=0.5, clip=5.0)
print(f"torch {torch.__version__} | device={DEVICE}")

torch 2.14.1+cpu | device=cpu


## 1. Data

In [3]:
df = pd.read_csv("secom_telemetry_modified.csv", parse_dates=["timestamp"])
df = df.sort_values("timestamp", kind="stable").reset_index(drop=True)

SENSOR_COLS = [c for c in df.columns if c.startswith("sensor_")]
X_raw = df[SENSOR_COLS].to_numpy(dtype=np.float64)          # [N, 590]
y_raw = (df["label"].to_numpy() == 1).astype(np.int64)      # 1 = defective
ts    = df["timestamp"].to_numpy()

print(f"X: {X_raw.shape} | defect rate: {y_raw.mean():.2%} "
      f"({y_raw.sum()} fail / {(1 - y_raw).sum()} pass)")
print(f"missing cells: {np.isnan(X_raw).mean():.2%} | {df.timestamp.min()} -> {df.timestamp.max()}")

X: (1567, 590) | defect rate: 6.64% (104 fail / 1463 pass)
missing cells: 4.54% | 2008-07-19 11:55:00 -> 2008-10-17 06:07:00


In [4]:
leg   = df["sensor_0"].eq(0.0).to_numpy()
codes = df["sensor_1"].where(leg)

print(f"PS-L wafers (sensor_0 == 0.0): {leg.sum()} | sensor_1 on them: {codes.value_counts().to_dict()}")
print(f"firmware codes on other rows: {df.loc[~leg, 'sensor_1'].isin([100.0, 200.0, 300.0]).sum()} | "
      f"genuine sensor_1 range {df.loc[~leg, 'sensor_1'].min():.1f} .. {df.loc[~leg, 'sensor_1'].max():.1f}")
print(f"defect rate PS-L {y_raw[leg].mean():.2%} ({y_raw[leg].sum()}/{leg.sum()}) | other stations {y_raw[~leg].mean():.2%}")
assert set(codes.dropna()) <= {100.0, 200.0, 300.0}

PS-L wafers (sensor_0 == 0.0): 146 | sensor_1 on them: {100.0: 61, 300.0: 50, 200.0: 35}
firmware codes on other rows: 0 | genuine sensor_1 range 2158.8 .. 2846.4
defect rate PS-L 0.00% (0/146) | other stations 7.32%


## 2 · Preprocessing

Split by time first. Every fitted step (column filter, imputer, scaler, threshold) only sees rows before the window it is applied to. The PS-L rewrite is a fixed per-row rule with nothing fitted, so it can run on the whole table.

In [5]:
def time_cuts(ts, fracs):
    n, cuts = len(ts), []
    for f in np.cumsum(fracs)[:-1]:
        i = int(f * n)
        while i < n and ts[i] == ts[i - 1]:     # lots with the same start time stay on one side
            i += 1
        cuts.append(i)
    return np.split(np.arange(n), cuts)

idx_tr, idx_va, idx_te = time_cuts(ts, HP["split"])
for name, ix in [("train", idx_tr), ("val", idx_va), ("test", idx_te)]:
    print(f"{name:5s} {len(ix):4d} rows | {y_raw[ix].sum():2d} fail ({y_raw[ix].mean():.2%}) | "
          f"{df.timestamp[ix[0]]} -> {df.timestamp[ix[-1]]}")

train  940 rows | 76 fail (8.09%) | 2008-07-19 11:55:00 -> 2008-09-20 05:34:00
val    313 rows | 11 fail (3.51%) | 2008-09-20 06:08:00 -> 2008-10-02 19:25:00
test   314 rows | 17 fail (5.41%) | 2008-10-02 20:54:00 -> 2008-10-17 06:07:00


In [6]:
# PS-L: the sentinel and the firmware codes are not measurements. Blank them in the
# continuous block and carry the code as a one-hot that skips imputation and scaling.
cont = df[SENSOR_COLS].copy()
cont.loc[leg, ["sensor_0", "sensor_1"]] = np.nan
X_cont = cont.to_numpy(dtype=np.float64)
X_cat  = np.stack([(codes == c).to_numpy() for c in (100.0, 200.0, 300.0)], axis=1).astype(np.float64)


def prep(fit_rows, rows, max_missing=HP["max_missing"], clip=HP["clip"]):
    Xf = X_cont[fit_rows]
    keep = (np.isnan(Xf).mean(axis=0) < max_missing) & (np.nanstd(Xf, axis=0) > 0)
    imputer = SimpleImputer(strategy="mean").fit(Xf[:, keep])
    scaler  = StandardScaler().fit(imputer.transform(Xf[:, keep]))
    out = []
    for ix in rows:
        xc = X_cont[ix][:, keep]
        xs = scaler.transform(imputer.transform(xc))
        if clip:
            xs = np.clip(xs, -clip, clip)
        X = np.hstack([xs, X_cat[ix]]).astype(np.float32)
        M = np.hstack([~np.isnan(xc), np.ones_like(X_cat[ix], dtype=bool)]).astype(np.float32)
        out.append((X, M))
    return out, (keep, imputer, scaler)


((X_train, M_train), (X_val, M_val), (X_test, M_test)), (keep, imputer, scaler) = prep(idx_tr, [idx_tr, idx_va, idx_te])
y_train, y_val, y_test = y_raw[idx_tr], y_raw[idx_va], y_raw[idx_te]

Xf = X_cont[idx_tr]
print(f"kept {keep.sum()} / {len(keep)} sensors | constant in train: {(np.nanstd(Xf, axis=0) == 0).sum()} | "
      f">= 50% missing in train: {(np.isnan(Xf).mean(axis=0) >= HP['max_missing']).sum()}")
print(f"train {X_train.shape} | val {X_val.shape} | test {X_test.shape} | masked (not measured) cells in train: {1 - M_train.mean():.2%}")

kept 444 / 590 sensors | constant in train: 122 | >= 50% missing in train: 24
train (940, 447) | val (313, 447) | test (314, 447) | masked (not measured) cells in train: 2.03%


## 3 · Tensors & loaders

In [7]:
F = X_train.shape[1]

def make_loader(X, M, y, shuffle, seed=SEED):
    # no channel axis: the model flattens to [B, F], so the loss target must be [B, F] too
    t = [torch.from_numpy(X), torch.from_numpy(M), torch.from_numpy(y).float()]
    g = torch.Generator().manual_seed(seed)
    return DataLoader(TensorDataset(*t), batch_size=HP["batch_size"],
                      shuffle=shuffle, generator=g if shuffle else None, drop_last=shuffle)

xb, mb, yb = next(iter(make_loader(X_train, M_train, y_train, shuffle=True)))
print(f"F = {F} | batch x {tuple(xb.shape)} | mask {tuple(mb.shape)} | y {tuple(yb.shape)}")

F = 447 | batch x (64, 447) | mask (64, 447) | y (64,)


## 4 · Model: supervised autoencoder (unchanged)

In [8]:
class WaferSAE(nn.Module):
    def __init__(self, d_in: int, d_hid: int, d_lat: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, d_hid), nn.LayerNorm(d_hid), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(d_hid, d_hid // 2), nn.LayerNorm(d_hid // 2), nn.GELU(),
            nn.Linear(d_hid // 2, d_lat),
        )
        self.decoder = nn.Sequential(
            nn.Linear(d_lat, d_hid // 2), nn.GELU(),
            nn.Linear(d_hid // 2, d_hid), nn.GELU(),
            nn.Linear(d_hid, d_in),
        )
        self.head = nn.Linear(d_lat, 1)

    def forward(self, x):
        x = x.flatten(start_dim=1)                       # tolerate [B, F] or [B, C, F]
        z = self.encoder(x)
        return self.decoder(z), self.head(z).squeeze(-1), z

print(f"parameters: {sum(p.numel() for p in WaferSAE(F, HP['hidden'], HP['latent']).parameters()):,}")

parameters: 304,640


## 5 · Training

Class balance comes from `pos_weight` instead of SMOTE (same CV score, see 8.4, and no synthetic wafers). The reconstruction loss only counts cells that were actually measured, so the decoder is never trained to reproduce an imputed value or a PS-L blank.

In [9]:
def train_sae(X, M, y, seed=SEED, log=True, broadcast_bug=False):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = WaferSAE(X.shape[1], HP["hidden"], HP["latent"]).to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=HP["lr"], weight_decay=HP["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=HP["epochs"])
    bce   = nn.BCEWithLogitsLoss(pos_weight=torch.tensor((1 - y.mean()) / y.mean(), device=DEVICE))
    loader = make_loader(X, M, y, shuffle=True, seed=seed)
    history = {"total": [], "recon": [], "clf": []}

    for epoch in range(1, HP["epochs"] + 1):
        model.train()
        acc = dict(total=0.0, recon=0.0, clf=0.0); n_batches = 0

        for batch_x, batch_m, batch_y in loader:
            batch_x, batch_m, batch_y = batch_x.to(DEVICE), batch_m.to(DEVICE), batch_y.to(DEVICE)

            reconstructed, logits, _ = model(batch_x)

            if broadcast_bug:      # the original [B, 1, F] target, only used for the ablations in section 8
                recon_loss = torch.mean((reconstructed - batch_x.unsqueeze(1)) ** 2)
            else:
                assert reconstructed.shape == batch_x.shape and logits.shape == batch_y.shape
                recon_loss = ((reconstructed - batch_x) ** 2 * batch_m).sum() / batch_m.sum()
            clf_loss   = bce(logits, batch_y)
            loss       = recon_loss + HP["lambda_clf"] * clf_loss

            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()

            acc["total"] += loss.item(); acc["recon"] += recon_loss.item(); acc["clf"] += clf_loss.item()
            n_batches += 1

        sched.step()
        for k in acc:
            history[k].append(acc[k] / n_batches)
        if log and (epoch == 1 or epoch % 5 == 0):
            print(f"epoch {epoch:02d}/{HP['epochs']} | total {history['total'][-1]:.4f} "
                  f"| recon {history['recon'][-1]:.4f} | clf {history['clf'][-1]:.4f}")
    return model, history


model, history = train_sae(X_train, M_train, y_train)

epoch 01/40 | total 2.0174 | recon 0.7563 | clf 1.2611
epoch 05/40 | total 0.6884 | recon 0.6642 | clf 0.0242


epoch 10/40 | total 0.5772 | recon 0.5753 | clf 0.0019


epoch 15/40 | total 0.5204 | recon 0.5190 | clf 0.0014


epoch 20/40 | total 0.4851 | recon 0.4835 | clf 0.0016


epoch 25/40 | total 0.4626 | recon 0.4616 | clf 0.0010


epoch 30/40 | total 0.4487 | recon 0.4474 | clf 0.0013


epoch 35/40 | total 0.4434 | recon 0.4425 | clf 0.0009


epoch 40/40 | total 0.4428 | recon 0.4421 | clf 0.0007


## 6 · Evaluation

Test is the last 20% of the timeline at its own prevalence, no resampling. The threshold is the max-F1 point on the validation window (also natural prevalence) and is then applied to test once. AP (area under the PR curve) is the main threshold-free number, since ROC-AUC looks flattering when 94% of wafers pass. The AP of a random scorer equals the prevalence.

In [10]:
def predict(model, X):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(torch.from_numpy(X).to(DEVICE))[1]).cpu().numpy()

def f1_threshold(y, p):
    prec, rec, thr = precision_recall_curve(y, p)
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-12)
    return thr[np.argmax(f1[:-1])]

def report(y, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(prevalence=y.mean(), AP=average_precision_score(y, p), ROC_AUC=roc_auc_score(y, p),
                F1=f1_score(y, pred, zero_division=0), precision=precision_score(y, pred, zero_division=0),
                recall=recall_score(y, pred, zero_division=0), TP=tp, FP=fp, FN=fn, TN=tn)

p_val, p_test = predict(model, X_val), predict(model, X_test)
thr = f1_threshold(y_val, p_val)
res = report(y_test, p_test, thr)

print(f"threshold (max F1 on val): {thr:.2e}")
pd.DataFrame([report(y_val, p_val, thr), res], index=["val", "test"]).round(3)

threshold (max F1 on val): 2.20e-05


,prevalence,AP,ROC_AUC,F1,precision,recall,TP,FP,FN,TN
val,0.035,0.034,0.480,0.082,0.043,0.909,10,223,1,79
test,0.054,0.070,0.577,0.124,0.068,0.765,13,179,4,118


In [11]:
pi = y_raw.mean()
tpr, fpr = res["TP"] / (res["TP"] + res["FN"]), res["FP"] / (res["FP"] + res["TN"])
prec_adj = tpr * pi / (tpr * pi + fpr * (1 - pi))
print(f"test window prevalence {y_test.mean():.2%}. Re-weighted to the dataset-wide {pi:.2%}: "
      f"precision {prec_adj:.3f} | recall {tpr:.3f} | F1 {2 * prec_adj * tpr / (prec_adj + tpr):.3f}")

rng, boot = np.random.default_rng(SEED), []
for _ in range(2000):
    b = rng.integers(0, len(y_test), len(y_test))
    if y_test[b].sum() == 0:
        continue
    boot.append([average_precision_score(y_test[b], p_test[b]), roc_auc_score(y_test[b], p_test[b]),
                 f1_score(y_test[b], (p_test[b] >= thr).astype(int), zero_division=0)])
lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
for i, k in enumerate(["AP", "ROC_AUC", "F1"]):
    print(f"{k:8s} {res[k]:.3f}   95% bootstrap CI [{lo[i]:.3f}, {hi[i]:.3f}]")

test window prevalence 5.41%. Re-weighted to the dataset-wide 6.64%: precision 0.083 | recall 0.765 | F1 0.149


AP       0.070   95% bootstrap CI [0.042, 0.137]
ROC_AUC  0.577   95% bootstrap CI [0.444, 0.710]
F1       0.124   95% bootstrap CI [0.067, 0.186]


In [12]:
rows = []
for s in [1337, 1, 2, 3, 4]:
    m, _ = train_sae(X_train, M_train, y_train, seed=s, log=False)
    rows.append({"seed": s, **report(y_test, predict(m, X_test), f1_threshold(y_val, predict(m, X_val)))})
seeds = pd.DataFrame(rows).set_index("seed")
print(seeds[["AP", "ROC_AUC", "F1", "precision", "recall", "TP", "FP", "FN"]].round(3))
print(seeds[["AP", "ROC_AUC", "F1"]].agg(["mean", "std"]).round(3))

         AP  ROC_AUC     F1  precision  recall  TP   FP  FN
seed                                                       
1337  0.070    0.577  0.124      0.068   0.765  13  179   4
1     0.087    0.622  0.128      0.074   0.471   8  100   9
2     0.080    0.613  0.132      0.075   0.588  10  124   7
3     0.095    0.646  0.151      0.092   0.412   7   69  10
4     0.091    0.630  0.115      0.063   0.706  12  179   5
         AP  ROC_AUC     F1
mean  0.084    0.618  0.130
std   0.010    0.026  0.013


In [13]:
# same split, same features, threshold picked on val the same way
base = {"SAE (this notebook)": res}
lr = LogisticRegression(C=0.01, class_weight="balanced", max_iter=5000).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=500, min_samples_leaf=3, class_weight="balanced_subsample",
                            n_jobs=4, random_state=SEED).fit(X_train, y_train)
for name, clf in [("logistic regression", lr), ("random forest", rf)]:
    base[name] = report(y_test, clf.predict_proba(X_test)[:, 1], f1_threshold(y_val, clf.predict_proba(X_val)[:, 1]))
base["flag every wafer"] = report(y_test, np.ones(len(y_test)), 0.5)
pd.DataFrame.from_dict(base, orient="index")[["AP", "ROC_AUC", "F1", "precision", "recall", "TP", "FP", "FN"]].round(3)

,AP,ROC_AUC,F1,precision,recall,TP,FP,FN
SAE (this notebook),0.070,0.577,0.124,0.068,0.765,13,179,4
logistic regression,0.088,0.638,0.117,0.063,0.765,13,193,4
random forest,0.077,0.608,0.145,0.083,0.588,10,111,7
flag every wafer,0.054,0.500,0.103,0.054,1.000,17,297,0


## 7 · Release gate

In [14]:
GATE_F1 = 0.90
f1 = res["F1"]
verdict = "PASSED" if f1 >= GATE_F1 else "FAILED"
print(f"Release gate (F1 ≥ {GATE_F1}) on the time-ordered test set: {f1:.4f} -> {verdict}")

if f1 >= GATE_F1:
    os.makedirs("artifacts", exist_ok=True)
    torch.save(model.state_dict(), "artifacts/wafer_sae.pt")
    joblib.dump({"keep": keep, "imputer": imputer, "scaler": scaler, "clip": HP["clip"], "threshold": thr},
                "artifacts/preprocessor.joblib")
else:
    print("not exporting artifacts, see AUDIT.md")

Release gate (F1 ≥ 0.9) on the time-ordered test set: 0.1244 -> FAILED
not exporting artifacts, see AUDIT.md


## 8 · Audit evidence

### 8.1 Reconstruction loss broadcasts to [B, B, F]

The original loader added a channel axis (`[B, 1, F]`) and `forward` flattens it away, so `reconstructed - batch_x` was `[B, F] - [B, 1, F]`.

In [15]:
xb = torch.from_numpy(X_train[:64]).unsqueeze(1)          # what the original loader produced
rec, logits, _ = WaferSAE(F, HP["hidden"], HP["latent"])(xb)
print("batch_x", tuple(xb.shape), "| reconstructed", tuple(rec.shape), "| reconstructed - batch_x", tuple((rec - xb).shape))

rec = rec.detach().requires_grad_()
g_bug  = torch.autograd.grad(torch.mean((rec - xb) ** 2), rec)[0]
g_mean = torch.autograd.grad(torch.mean((rec - xb.squeeze(1).mean(0)) ** 2), rec)[0]
print(f"grad of broadcast loss vs grad of MSE to the batch mean: max abs diff {(g_bug - g_mean).abs().max():.2e}")

batch_x (64, 1, 447) | reconstructed (64, 447) | reconstructed - batch_x (64, 64, 447)
grad of broadcast loss vs grad of MSE to the batch mean: max abs diff 2.18e-11


So every reconstruction is pulled toward the batch mean, not toward its own input. Training both versions on the same data:

In [16]:
m_bug, h_bug = train_sae(X_train, M_train, y_train, broadcast_bug=True, log=False)

for name, m, h in [("broadcast (original)", m_bug, h_bug), ("fixed", model, history)]:
    m.eval()
    with torch.no_grad():
        r = m(torch.from_numpy(X_val))[0].numpy()
    err = ((r - X_val) ** 2 * M_val).sum(1) / M_val.sum(1)
    print(f"{name:21s} train recon {h['recon'][0]:.3f} -> {h['recon'][-1]:.3f} | val per-wafer MSE {err.mean():.3f} | "
          f"reconstruction spread across wafers {r.std(0).mean():.3f} | val AP {average_precision_score(y_val, predict(m, X_val)):.3f}")
print(f"input spread across wafers: {X_val.std(0).mean():.3f}")

broadcast (original)  train recon 0.744 -> 0.744 | val per-wafer MSE 0.847 | reconstruction spread across wafers 0.004 | val AP 0.036
fixed                 train recon 0.756 -> 0.442 | val per-wafer MSE 0.642 | reconstruction spread across wafers 0.384 | val AP 0.034
input spread across wafers: 0.765


### 8.2 Leakage ablation

Starting from the original pipeline and fixing one thing at a time. Steps 0 to 3 use a random 80/20 split like the original; from step 4 on the split is the time-ordered one above. Steps 0 to 5 keep the original preprocessing (global mean imputer and scaler over raw sensors, SMOTE).

In [17]:
def orig_prep(fit_rows, rows):
    imp = SimpleImputer(strategy="mean", keep_empty_features=True).fit(X_raw[fit_rows])
    sc  = StandardScaler().fit(imp.transform(X_raw[fit_rows]))
    return [sc.transform(imp.transform(X_raw[r])).astype(np.float32) for r in rows]

def fit_smote(X, y, bug):
    Xb, yb = SMOTE(sampling_strategy="auto", k_neighbors=5, random_state=SEED).fit_resample(X, y)
    m, _ = train_sae(Xb.astype(np.float32), np.ones_like(Xb, dtype=np.float32), yb, broadcast_bug=bug, log=False)
    return m

ladder = {}
all_rows = np.arange(len(y_raw))

# 0-1: SMOTE on everything, then a random split (as in the original)
(X_all,) = orig_prep(all_rows, [all_rows])
X_bal, y_bal = SMOTE(sampling_strategy="auto", k_neighbors=5, random_state=SEED).fit_resample(X_all, y_raw)
is_syn = np.arange(len(y_bal)) >= len(y_raw)
i_tr, i_te = train_test_split(np.arange(len(y_bal)), test_size=0.2, stratify=y_bal, random_state=SEED)
for step, bug in [("0 original", True), ("1 + fix loss broadcast", False)]:
    m, _ = train_sae(X_bal[i_tr].astype(np.float32), np.ones((len(i_tr), X_bal.shape[1]), np.float32), y_bal[i_tr], broadcast_bug=bug, log=False)
    ladder[step] = report(y_bal[i_te], predict(m, X_bal[i_te].astype(np.float32)), 0.5)

# 2-3: split real wafers first, SMOTE only the training part, test at natural prevalence
r_tr, r_te = train_test_split(all_rows, test_size=0.2, stratify=y_raw, random_state=SEED)
Xa_tr, Xa_te = X_all[r_tr], X_all[r_te]
ladder["2 + SMOTE after split (train only)"] = report(y_raw[r_te], predict(fit_smote(Xa_tr, y_raw[r_tr], False), Xa_te), 0.5)
Xb_tr, Xb_te = orig_prep(r_tr, [r_tr, r_te])
ladder["3 + imputer/scaler fit on train only"] = report(y_raw[r_te], predict(fit_smote(Xb_tr, y_raw[r_tr], False), Xb_te), 0.5)

# 4-5: time-ordered split
Xc_tr, Xc_va, Xc_te = orig_prep(idx_tr, [idx_tr, idx_va, idx_te])
m4 = fit_smote(Xc_tr, y_train, False)
ladder["4 + time-ordered split"] = report(y_test, predict(m4, Xc_te), 0.5)
ladder["5 + threshold from val"] = report(y_test, predict(m4, Xc_te), f1_threshold(y_val, predict(m4, Xc_va)))
ladder["6 final pipeline (sections 2-6)"] = res

pd.DataFrame.from_dict(ladder, orient="index")[["prevalence", "AP", "ROC_AUC", "F1", "precision", "recall", "TP", "FP", "FN"]].round(3)

,prevalence,AP,ROC_AUC,F1,precision,recall,TP,FP,FN
0 original,0.500,1.000,1.000,0.992,0.983,1.000,293,5,0
1 + fix loss broadcast,0.500,0.999,0.999,0.987,0.973,1.000,293,8,0
2 + SMOTE after split (train only),0.067,0.088,0.567,0.067,0.111,0.048,1,8,20
3 + imputer/scaler fit on train only,0.067,0.095,0.577,0.071,0.143,0.048,1,6,20
4 + time-ordered split,0.054,0.084,0.589,0.000,0.000,0.000,0,1,17
5 + threshold from val,0.054,0.084,0.589,0.098,0.056,0.412,7,119,10
6 final pipeline (sections 2-6),0.054,0.070,0.577,0.124,0.068,0.765,13,179,4


Why step 0 looks perfect: SMOTE ran before the split, so test defects are either synthetic points made from training defects, or real defects whose synthetic offspring ended up in train.

In [18]:
te_fail = i_te[y_bal[i_te] == 1]
d, j = NearestNeighbors(n_neighbors=1).fit(X_bal[i_tr]).kneighbors(X_bal[te_fail])
d, j = d.ravel(), j.ravel()
d_clean = NearestNeighbors(n_neighbors=1).fit(X_all[r_tr]).kneighbors(X_all[r_te][y_raw[r_te] == 1])[0].ravel()

print(f"test defects: {len(te_fail)} | synthetic: {is_syn[te_fail].sum()} ({is_syn[te_fail].mean():.1%}) | real: {(~is_syn[te_fail]).sum()}")
print(f"nearest training row is a SMOTE point for {is_syn[i_tr][j].mean():.1%} of test defects")
print(f"median distance to nearest training row: synthetic test defects {np.median(d[is_syn[te_fail]]):.2f} | "
      f"real test defects {np.median(d[~is_syn[te_fail]]):.2f}")
print(f"clean split for reference (real defect -> nearest real training wafer): {np.median(d_clean):.2f}")
print(f"test-set prevalence {y_bal[i_te].mean():.1%} vs natural {y_raw.mean():.1%}")

test defects: 293 | synthetic: 274 (93.5%) | real: 19
nearest training row is a SMOTE point for 92.8% of test defects
median distance to nearest training row: synthetic test defects 2.37 | real test defects 0.99
clean split for reference (real defect -> nearest real training wafer): 18.46
test-set prevalence 50.0% vs natural 6.6%


### 8.3 PS-L sentinel and firmware codes in the original preprocessing

In [19]:
s0, s1 = df["sensor_0"], df["sensor_1"]
print(pd.DataFrame({"sensor_0 all rows": [s0.mean(), s0.std()], "sensor_0 genuine": [s0[~leg].mean(), s0[~leg].std()],
                    "sensor_1 all rows": [s1.mean(), s1.std()], "sensor_1 genuine": [s1[~leg].mean(), s1[~leg].std()]},
                   index=["mean", "std"]).round(1))

(X_o,) = orig_prep(all_rows, [all_rows])
print(f"\noriginal scaler, sensor_0 on PS-L rows: z = {np.unique(X_o[leg, 0].round(2))}")
print(f"original scaler, sensor_1 codes 100/200/300: z = {[round(float(X_o[leg & (s1 == c).to_numpy(), 1][0]), 2) for c in (100.0, 200.0, 300.0)]}")
print(f"genuine variation squashed by {s0.std() / s0[~leg].std():.1f}x (sensor_0) and {s1.std() / s1[~leg].std():.1f}x (sensor_1)")
nan0 = s0.isna().to_numpy()
print(f"{nan0.sum()} genuinely missing sensor_0 values imputed to {s0.mean():.1f}, which is "
      f"{(s0.mean() - s0[~leg].mean()) / s0[~leg].std():.1f} genuine std from the real mean")
print(f"rule 'scaled sensor_0 < -2' picks out {(X_o[:, 0] < -2).sum()} wafers, all PS-L, defect rate {y_raw[X_o[:, 0] < -2].mean():.0%}")

      sensor_0 all rows  sensor_0 genuine  sensor_1 all rows  sensor_1 genuine
mean             2732.1            3014.0             2280.5            2495.7
std               880.7              74.3              675.3              79.8

original scaler, sensor_0 on PS-L rows: z = [-3.11]
original scaler, sensor_1 codes 100/200/300: z = [-3.23, -3.09, -2.94]
genuine variation squashed by 11.8x (sensor_0) and 8.5x (sensor_1)
6 genuinely missing sensor_0 values imputed to 2732.1, which is -3.8 genuine std from the real mean
rule 'scaled sensor_0 < -2' picks out 146 wafers, all PS-L, defect rate 0%


### 8.4 Design choices (walk-forward CV inside the train + val window only)

Four expanding-window folds: train on everything before each 10% block from 40% to 80% of the timeline, score the block. Test rows are not touched. Mean over 3 seeds of the per-block AP and ROC-AUC.

In [20]:
Xf, Xl = X_cont[idx_tr], X_cont[np.r_[idx_va, idx_te]]
flat = (np.nanstd(Xf, axis=0) == 0) & (np.nanstd(Xl, axis=0) > 0)
((_, _), (Xu, _)), _ = prep(idx_tr, [idx_tr, idx_te], max_missing=1.0, clip=None)
print(f"sensors constant in train that move later: {flat.sum()} | "
      f"max |z| on test with a train-fit scaler and no clip: {np.abs(Xu[:, :-3]).max():.0f} | "
      f"test cells with |z| > 5: {(np.abs(Xu[:, :-3]) > 5).mean():.2%}")

sensors constant in train that move later: 6 | max |z| on test with a train-fit scaler and no clip: 37231 | test cells with |z| > 5: 1.11%


In [21]:
n = len(y_raw)
starts = [time_cuts(ts, (f, 1 - f))[1][0] for f in (0.4, 0.5, 0.6, 0.7, 0.8)]
blocks = list(zip(starts[:-1], starts[1:]))
print("defects per block:", [int(y_raw[a:b].sum()) for a, b in blocks], "| mean prevalence", round(np.mean([y_raw[a:b].mean() for a, b in blocks]), 3))

def cv_score(max_missing, clip, smote=False, seeds=(1337, 1, 2)):
    out = []
    for s in seeds:
        for a, b in blocks:
            tr, ev = np.arange(a), np.arange(a, b)
            ((Xt, Mt), (Xe, _)), _ = prep(tr, [tr, ev], max_missing, clip)
            yt = y_raw[tr]
            if smote:
                Xt, yt = SMOTE(k_neighbors=5, random_state=s).fit_resample(Xt, yt)
                Mt = np.ones_like(Xt, dtype=np.float32)
            m, _ = train_sae(Xt.astype(np.float32), Mt, yt, seed=s, log=False)
            p = predict(m, Xe)
            out.append([average_precision_score(y_raw[ev], p), roc_auc_score(y_raw[ev], p)])
    return np.mean(out, axis=0).round(3)

cv = {"all sensors, no clip": cv_score(1.0, None),
      "+ clip z to ±5": cv_score(1.0, 5.0),
      "+ drop ≥50% missing (chosen)": cv_score(0.5, 5.0),
      "same, SMOTE instead of pos_weight/mask": cv_score(0.5, 5.0, smote=True)}
pd.DataFrame(cv, index=["AP", "ROC_AUC"]).T

defects per block: [2, 9, 2, 9] | mean prevalence 0.035


,AP,ROC_AUC
"all sensors, no clip",0.045,0.483
+ clip z to ±5,0.054,0.547
+ drop ≥50% missing (chosen),0.058,0.592
"same, SMOTE instead of pos_weight/mask",0.058,0.585
